In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import scipy.io as sio
from dataclasses import dataclass
from typing import List, Tuple
import os
from dotenv import load_dotenv
load_dotenv()
import tidy3d as td
from tidy3d import web
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from natsort import natsorted
import numpy as np
import matplotlib.animation as animation
import xarray as xr
import h5py
import imageio
import matplotlib
import gc
import sys
import io
import matplotlib.colors as mcolors
import matplotlib.patches as patches
from scipy.optimize import curve_fit
import scipy.integrate
import re
import scipy.ndimage

# Assuming /AutomationModule is in the root directory of your project
sys.path.append(os.path.abspath(rf'H:\codes\tidy3d'))

from AutomationModule import * 

import AutomationModule as AM
plt.rcParams.update({'font.size': 22})  

tidy3dAPI = os.environ["API_TIDY3D_KEY"]
plt.rc('font', family='Arial')


In [2]:
import time 
# time.sleep(3600)

In [3]:
file_data = "./data/slab_250x250x32/Transmission/LSU_20260804_slab_250x250x32_Transmission_background_n_1.00.h5"

data = AM.read_hdf5_as_dict(file_data)

In [4]:
for path_direction in [
                  rf"../../../../data/20260813_Beam_Spreading_250_250_32_transmission"
                       ]:
      folder_path = f"{path_direction}"
      
     
      for dirpath, dirnames, filenames in os.walk(folder_path):
            for filename in natsorted(filenames):
                  try:
                        n = re.search(r'n_(\d+(?:\.\d+)?)', filename).group(1)
                  except:
                        n = re.search(r'n_(\d+(?:\.\d+)?)', dirpath).group(1)
                  if str(n) not in data.keys():
                        data[str(n)] = {}
                  else:
                        continue
                  if os.path.isfile(os.path.join(dirpath, filename)):
                        file=os.path.join(dirpath, filename)
                        structure_1 = AM.loadFromFile(key = tidy3dAPI, file_path=file,get_ref=True)
                        flux_exit = structure_1.sim_data.monitor_data["transmission_monitor_exit"].flux.values
                        flux_exit_ref = structure_1.sim_data0.monitor_data["transmission_monitor_exit"].flux.values
                        transmission = flux_exit / flux_exit_ref
                        data[str(n)] = {
                              "Transmission_Exit":transmission,
                              "f":structure_1.sim_data.monitor_data["transmission_monitor_exit"].flux.f.values
                        }
                        del structure_1

                 
                

Configured successfully.


15:10:55 W. Europe Daylight Time WARNING: Structure at 'structures[1]' has      
                                 bounds that extend exactly to simulation edges.
                                 This can cause unexpected behavior. If         
                                 intending to extend the structure to infinity  
                                 along one dimension, use td.inf as a size      
                                 variable instead to make this explicit.        

                                 WARNING: Suppressed 1 WARNING message.         

15:16:51 W. Europe Daylight Time Billed flex credit cost: 31.287.

                                 Note: the task cost pro-rated due to early     
                                 shutoff was below the minimum threshold, due to
                                 fast shutoff. Decreasing the simulation        
                                 'run_time' should decrease the estimated, and  
                                 correspondingly the billed cost of such tasks.

In [5]:
os.makedirs("./data/slab_250x250x32/Transmission",exist_ok=True)
AM.create_hdf5_from_dict(data,file_data)